# DINOv3 on sidewalk-data-3: hazard detection and four tag groups

Use a **Colab GPU runtime (T4 is sufficient)**. Run the cells in order and mount Google Drive.
This notebook is separate from `dino_ft_A_report_rerun.ipynb` and uses the new dataset's saved labels.

**Before Run All:** copy the server archive `/data/datasets/sidewalk-data-3.tar` to
`MyDrive/MLiP/datasets/sidewalk-data-3.tar` (or configure the Hugging Face option below).
The archive is about 2.47 GB. Supply an `HF_TOKEN` with access to
`facebook/dinov3-vitb16-pretrain-lvd1689m` through Colab Secrets, the environment, or the hidden prompt.
No token is embedded or saved in this notebook.

**Experiment:** frozen DINOv3 ViT-B/16, CLS features, a linear layer with five independent logits:
one hazard score and four tag-group scores. The hazard loss uses all images; group loss uses only
true hazards. Images can have multiple groups. The classes/tags are read from the dataset, never
inferred from the old Obstacle/SurfaceProblem names.

**Selection:**
1. Across the three seeds, maximize mean validation **precision@46**; break ties by the best mean
   validation **hazard balanced accuracy**, choosing the hazard threshold from `[0.3, 0.5, 0.7]`.
2. Independently choose the configuration with best mean validation **macro tag-group balanced
   accuracy**, on **all true hazards**, including those missed by stage 1. For each of the four
   groups, BA = (sensitivity + specificity)/2; macro BA is their unweighted mean. Choose each
   group's threshold from the same grid using validation means over seeds.
3. Save all three replicas of each winning configuration. For a single deployable model per task,
   choose the best validation replica within that configuration using the same objective; ties
   use the smaller seed. Report this deployment model separately from the three-seed mean.

The two training orders, seven learning rates, two weight decays and three seeds give **84 trials**,
not 168. Each trial trains all five linear outputs. Compare final checkpoints after 5,000 Pitt updates,
or 5,000 Other + 5,000 Pitt updates. Threshold searches do not add training runs. Test performance
is computed only after selection is frozen; it never affects the winning configurations, seeds or thresholds.

All artifacts are saved to `MyDrive/MLiP/dino_sidewalk_data_3_runs/<run-id>/`.
Resume by setting `RESUME_RUN_DIR` to that directory. Completed trials and feature caches are reused;
an interrupted trial restarts. Resume requires the same code, settings and feature-extraction environment.


In [ ]:
%pip install -q "transformers==5.22.1" pandas matplotlib huggingface_hub pillow
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
import torch, transformers
assert torch.cuda.is_available(), 'Choose Runtime > Change runtime type > T4 GPU, then rerun.'
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
print('GPU:', torch.cuda.get_device_name(0), '| torch:', torch.__version__, '| transformers:', transformers.__version__)


## Configuration
The default uses the prepared archive in Google Drive. The alternative Hugging Face mode requires
the repository and archive filename for **dataset 3**, not the old merged-nulls dataset.
Do not change hyperparameters while resuming; use a fresh run directory instead.


In [ ]:
from pathlib import Path
DATA_SOURCE = 'drive'  # 'drive', 'huggingface', or 'existing' (already extracted under DATA_ROOT)
DRIVE_ARCHIVE = Path('/content/drive/MyDrive/MLiP/datasets/sidewalk-data-3.tar')
HF_DATA_REPO = ''       # Required only for DATA_SOURCE='huggingface'
HF_DATA_ARCHIVE = 'sidewalk-data-3.tar'
HF_DATA_REVISION = None # Resolved to an immutable commit on first download
DATA_ROOT = Path('/content/sidewalk-data-3')
DRIVE_ROOT = Path('/content/drive/MyDrive/MLiP/dino_sidewalk_data_3_runs')
RESUME_RUN_DIR = None
RUN_TEST = True
CACHE_BATCH_SIZE = 64

BACKBONE = 'facebook/dinov3-vitb16-pretrain-lvd1689m'
BACKBONE_REVISION = '5931719e67bbdb9737e363e781fb0c67687896bc'
EXPECTED_ARCHIVE_SHA256 = 'd6c88a2b770baefd9ee34b7c846fe8be573258ba6d6ec7cac27e90e2d9eabbbd'
EXPECTED_DATASET_HASHES = {'other-cities-train.csv': 'dc88d9a1fd7683af2f6d551922904035b871577a0b9da79dcc797b79fd5719d5', 'pitt-train.csv': '6fdc86fffcf59152187011a2dd55d5594e63f17ac7abb01529fdf4b79dc63082', 'pitt-val.csv': '32bbce7479f2f366aed2f008be127d7ba4c30fd35aa91feb5e5f704f090319bc', 'pitt-test.csv': 'bff64ae48ef512c0e722cf96cd77f06090411386ef0217242d39740d7600388a', 'label_schema.json': 'b429abdd786bc6c6f191b5fec0d0104171f2722f03ce819ea71f53bdeab911db', 'COMPLETE.json': '40a4b52bf4b07084e5fcb04d0e8f3b712cc7cef282861603a0d0ee58c753b827'}

SETTINGS = dict(
    train_plans={'pitt': [('pitt-train', 5000)],
                 'other_then_pitt': [('other-cities-train', 5000), ('pitt-train', 5000)]},
    learning_rates=[1e-5, 3e-5, 1e-4, 3e-4, 1e-3, 3e-3, 1e-2],
    weight_decays=[0.0, 0.01], seeds=[0, 1, 2], batch_size=64, eval_every=250,
    thresholds=[0.3, 0.5, 0.7], k=46,
)


## Implementation helpers
This collapsed cell contains the self-contained training, selection, and inference implementations.
They are also saved with the results. No companion file needs to be downloaded into Colab.


In [ ]:
CORE_SOURCE = '"""Frozen DINOv3 linear-head sweep and validation-only model selection for dataset 3."""\nfrom pathlib import Path\nfrom concurrent.futures import ThreadPoolExecutor\nfrom collections import defaultdict\nfrom itertools import product\nimport hashlib, io, json, os, shutil, tempfile, time\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\nfrom PIL import Image\n\nSPLITS = [\'other-cities-train\', \'pitt-train\', \'pitt-val\', \'pitt-test\']\nGROUPS = [\'surface_and_level_problems\', \'restricted_passage\', \'missing_tactile_warning\', \'ramp_points_into_traffic\']\nGROUP_COLUMNS = [\'group_\' + g for g in GROUPS]\nMEAN = [0.485, 0.456, 0.406]\nSTD = [0.229, 0.224, 0.225]\n\n\ndef plain(x):\n    if isinstance(x, Path): return str(x)\n    if isinstance(x, np.generic): return x.item()\n    if isinstance(x, np.ndarray): return x.tolist()\n    raise TypeError(type(x).__name__)\n\n\ndef canonical(x): return json.dumps(x, sort_keys=True, separators=(\',\', \':\'), default=plain)\ndef fingerprint(x): return hashlib.sha256(canonical(x).encode()).hexdigest()\ndef file_sha(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for b in iter(lambda: f.read(1024*1024), b\'\'): h.update(b)\n    return h.hexdigest()\n\n\ndef atomic_bytes(path, payload):\n    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(\'.\' + path.name + \'.tmp\')\n    tmp.write_bytes(payload); os.replace(tmp, path)\n\n\ndef save_json(path, value): atomic_bytes(path, (json.dumps(value, indent=2, default=plain, allow_nan=False)+\'\\n\').encode())\ndef save_csv(path, frame): atomic_bytes(path, frame.to_csv(index=False).encode())\ndef save_torch(path, value):\n    with tempfile.TemporaryDirectory() as temp:\n        local = Path(temp)/\'model.pt\'; torch.save(value, local)\n        path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)\n        tmp = path.with_name(\'.\'+path.name+\'.tmp\'); shutil.copyfile(local, tmp); os.replace(tmp, path)\ndef save_npz(path, **arrays):\n    buf = io.BytesIO(); np.savez_compressed(buf, **arrays); atomic_bytes(path, buf.getvalue())\n\n\ndef configs(settings):\n    return [dict(plan=p, lr=float(lr), weight_decay=float(wd), seed=int(seed), batch_size=settings[\'batch_size\'])\n            for p,lr,wd,seed in product(settings[\'train_plans\'], settings[\'learning_rates\'], settings[\'weight_decays\'], settings[\'seeds\'])]\n\ndef config_id(cfg): return fingerprint({k:v for k,v in cfg.items() if k!=\'seed\'})[:16]\ndef trial_id(cfg): return fingerprint(cfg)[:16]\n\n\ndef load_data(root, expected_hashes):\n    root = Path(root)\n    for filename, expected in expected_hashes.items():\n        if file_sha(root/filename) != expected: raise ValueError(f\'Dataset identity mismatch: {filename}\')\n    schema = json.loads((root/\'label_schema.json\').read_text())\n    assert schema[\'group_order\']==GROUPS and schema[\'group_columns\']==GROUP_COLUMNS\n    assert schema[\'multi_label\'] is True and schema[\'binary_labels\']=={\'non_hazard\':0,\'hazard\':1}\n    data = {}\n    for split in SPLITS:\n        df = pd.read_csv(root/(split+\'.csv\'), low_memory=False, keep_default_na=False)\n        assert df[\'sample_id\'].is_unique and df[\'image_path\'].is_unique\n        assert set(df[\'split\'])=={split}\n        y = df[\'hazard_label\'].to_numpy(dtype=np.int64)\n        yg = df[GROUP_COLUMNS].to_numpy(dtype=np.int64)\n        assert set(np.unique(y))=={0,1} and set(np.unique(yg))<= {0,1}\n        assert np.array_equal(y, yg.any(axis=1).astype(np.int64))\n        assert np.array_equal(df[\'class\'].to_numpy(), np.where(y, \'hazard\', \'non_hazard\'))\n        for g in GROUPS:\n            tagcols = schema[\'groups\'][g][\'positive_tags\']\n            expected = df[tagcols].astype(str).replace(\'\', \'0\').astype(float).eq(1).any(axis=1).to_numpy()\n            assert np.array_equal(expected, df[\'group_\'+g].to_numpy(dtype=bool))\n        assert not df.loc[df.is_manual_collection.astype(int)==1, \'hazard_label\'].astype(int).any()\n        # A missing class makes balanced accuracy ill-defined for the planned task.\n        assert ((yg[y==1].sum(0)>0)&(yg[y==1].sum(0)<int(y.sum()))).all()\n        assert all((root/p).is_file() and (root/p).resolve().is_relative_to(root.resolve()) for p in df.image_path)\n        data[split] = df\n    assert len(set().union(*(set(df.sample_id) for df in data.values()))) == sum(map(len,data.values()))\n    return data, schema\n\n\ndef targets(df):\n    return df.hazard_label.to_numpy(dtype=np.float32), df[GROUP_COLUMNS].to_numpy(dtype=np.float32)\n\n\ndef binary_metrics(y, pred):\n    y=np.asarray(y,dtype=bool); pred=np.asarray(pred,dtype=bool)\n    assert y.shape==pred.shape and y.ndim==1 and y.any() and (~y).any()\n    tp=int((y&pred).sum()); fn=int((y&~pred).sum()); tn=int((~y&~pred).sum()); fp=int((~y&pred).sum())\n    recall=tp/(tp+fn); specificity=tn/(tn+fp)\n    precision=tp/(tp+fp) if tp+fp else 0.\n    f1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.\n    return dict(tp=tp,fn=fn,tn=tn,fp=fp,support=tp+fn,negative_support=tn+fp,\n                precision=precision,recall=recall,specificity=specificity,f1=f1,\n                accuracy=(tp+tn)/len(y),balanced_accuracy=(recall+specificity)/2)\n\n\ndef hazard_metrics(y, p, threshold, sample_ids, k):\n    y=np.asarray(y,dtype=bool); p=np.asarray(p)\n    assert len(y)>=k>0 and p.shape==y.shape and np.isfinite(p).all()\n    # sample_id is the deterministic secondary key for equal confidence scores.\n    order=np.lexsort((np.asarray(sample_ids,dtype=str), -p))\n    hits=int(y[order[:k]].sum())\n    return {**binary_metrics(y,p>=threshold),\'topk_hits\':hits,\'precision_at_k\':hits/k,\'recall_at_k\':hits/int(y.sum()),\'k\':int(k)}\n\n\ndef group_metrics(yh, yg, p, thresholds, hazard_only=True):\n    yh=np.asarray(yh,dtype=bool); yg=np.asarray(yg,dtype=bool); p=np.asarray(p)\n    assert yg.shape==p.shape and yg.shape[1]==len(GROUPS) and np.isfinite(p).all()\n    mask=yh if hazard_only else np.ones(len(yh),dtype=bool)\n    pred=p[mask]>=np.asarray(thresholds)\n    per_group=[{\'tag_group\':g,**binary_metrics(yg[mask,j],pred[:,j])} for j,g in enumerate(GROUPS)]\n    summary={name:float(np.mean([r[name] for r in per_group])) for name in [\'balanced_accuracy\',\'precision\',\'recall\',\'f1\',\'accuracy\']}\n    summary[\'exact_match_accuracy\']=float(np.mean(np.all(pred==yg[mask],axis=1)))\n    summary[\'n_images\']=int(mask.sum())\n    return summary, per_group\n\n\ndef choose_threshold(scores):\n    """scores is a list of (threshold, mean validation BA); lower threshold breaks exact ties."""\n    return min(scores, key=lambda x:(-round(float(x[1]),12),float(x[0])))[0]\n\n\ndef select_models(entries, val_df, thresholds, seeds, k):\n    """Only final-checkpoint validation probabilities enter this function."""\n    yh,yg=targets(val_df); ids=val_df.sample_id.to_numpy(); bycfg=defaultdict(list)\n    for entry in entries: bycfg[config_id(entry[\'cfg\'])].append(entry)\n    summary=[]; hazard_table=[]; group_table=[]; selection_details={}\n    for cid, group in sorted(bycfg.items()):\n        group=sorted(group,key=lambda e:e[\'cfg\'][\'seed\'])\n        assert [e[\'cfg\'][\'seed\'] for e in group]==list(seeds), f\'Incomplete seed group: {cid}\'\n        hscores=[]; per_group_scores={g:[] for g in GROUPS}\n        for t in thresholds:\n            h=[]; gs=[[] for _ in GROUPS]\n            for e in group:\n                p=e[\'val_probabilities\']; assert p.shape==(len(val_df),5)\n                m=hazard_metrics(yh,p[:,0],t,ids,k); h.append(m[\'balanced_accuracy\'])\n                hazard_table.append(dict(config_id=cid,trial_id=e[\'trial_id\'],**e[\'cfg\'],threshold=t,**m))\n                _, gm=group_metrics(yh,yg,p[:,1:],[t]*4)\n                for j,g in enumerate(GROUPS):\n                    gs[j].append(gm[j][\'balanced_accuracy\'])\n                    group_table.append(dict(config_id=cid,trial_id=e[\'trial_id\'],**e[\'cfg\'],threshold=t,**gm[j]))\n            hscores.append((t,float(np.mean(h))))\n            for j,g in enumerate(GROUPS):per_group_scores[g].append((t,float(np.mean(gs[j]))))\n        ht=float(choose_threshold(hscores)); gt=[float(choose_threshold(per_group_scores[g])) for g in GROUPS]\n        hm=[hazard_metrics(yh,e[\'val_probabilities\'][:,0],ht,ids,k) for e in group]\n        gm=[group_metrics(yh,yg,e[\'val_probabilities\'][:,1:],gt)[0] for e in group]\n        base={key:value for key,value in group[0][\'cfg\'].items() if key!=\'seed\'}\n        row=dict(config_id=cid,**base,hazard_threshold=ht,group_thresholds=json.dumps(gt),\n                 topk_hits_sum=sum(m[\'topk_hits\'] for m in hm),\n                 precision_at_k_mean=float(np.mean([m[\'precision_at_k\'] for m in hm])),\n                 precision_at_k_std=float(np.std([m[\'precision_at_k\'] for m in hm],ddof=1)),\n                 hazard_ba_mean=float(np.mean([m[\'balanced_accuracy\'] for m in hm])),\n                 hazard_ba_std=float(np.std([m[\'balanced_accuracy\'] for m in hm],ddof=1)),\n                 group_macro_ba_mean=float(np.mean([m[\'balanced_accuracy\'] for m in gm])),\n                 group_macro_ba_std=float(np.std([m[\'balanced_accuracy\'] for m in gm],ddof=1)))\n        summary.append(row)\n        # Configuration selection uses seed means. A deployment replica is then\n        # selected from that winning configuration on validation alone.\n        hrep=min(zip(group,hm),key=lambda em:(-em[1][\'topk_hits\'],-round(em[1][\'balanced_accuracy\'],12),em[0][\'cfg\'][\'seed\']))[0]\n        grep=min(zip(group,gm),key=lambda em:(-round(em[1][\'balanced_accuracy\'],12),em[0][\'cfg\'][\'seed\']))[0]\n        selection_details[cid]=dict(summary=row,trial_ids=[e[\'trial_id\'] for e in group],\n                                   hazard_deployment_trial=hrep[\'trial_id\'],group_deployment_trial=grep[\'trial_id\'])\n    # Final deterministic ties prefer Pitt-only, lower LR, then lower weight decay.\n    fallback=lambda r:(0 if r[\'plan\']==\'pitt\' else 1,r[\'lr\'],r[\'weight_decay\'],r[\'config_id\'])\n    best_h=min(summary,key=lambda r:(-r[\'topk_hits_sum\'],-round(r[\'hazard_ba_mean\'],12),*fallback(r)))\n    best_g=min(summary,key=lambda r:(-round(r[\'group_macro_ba_mean\'],12),*fallback(r)))\n    selection=dict(hazard=selection_details[best_h[\'config_id\']],tag_groups=selection_details[best_g[\'config_id\']],\n                   k=k,group_order=GROUPS,validation_split=\'pitt-val\',test_used_for_selection=False,\n                   hazard_rule=\'Maximum mean validation precision@46 (integer summed top-k hits); tie-break maximum mean hazard BA at selected threshold.\',\n                   group_rule=\'Maximum mean macro group BA on ALL true validation hazards, each group threshold selected by its mean BA.\',\n                   seed_rule=\'Configuration selected across three seeds. Single deployment replica selected within winner by same validation objective; smaller seed breaks ties. All selected replicas also saved.\',\n                   threshold_grid=list(thresholds),checkpoint_rule=\'Final update only; no early-stopping/checkpoint search.\')\n    return selection,pd.DataFrame(summary),pd.DataFrame(hazard_table),pd.DataFrame(group_table)\n\n\ndef batch_indices(n, steps, batch_size, generator, device):\n    assert n>=batch_size\n    order=torch.randperm(n,generator=generator); offset=0\n    for _ in range(steps):\n        if offset+batch_size>n: order=torch.randperm(n,generator=generator);offset=0\n        yield order[offset:offset+batch_size].to(device)\n        offset+=batch_size\n\n\ndef task_losses(logits, yh, yg):\n    h=nn.functional.binary_cross_entropy_with_logits(logits[:,0],yh)\n    per_image=nn.functional.binary_cross_entropy_with_logits(logits[:,1:],yg,reduction=\'none\').mean(1)\n    g=(per_image*yh).sum()/yh.sum().clamp_min(1)\n    return h,g\n\n\n@torch.inference_mode()\ndef predict_head(state, features):\n    head=nn.Linear(features.shape[1],5).to(features.device)\n    head.load_state_dict(state);head.eval()\n    return head(features).sigmoid().float().cpu().numpy()\n\n\ndef train_trial(cfg, settings, features, data):\n    device=features[\'pitt-train\'].device\n    torch.manual_seed(cfg[\'seed\'])\n    generator=torch.Generator().manual_seed(cfg[\'seed\'])\n    head=nn.Linear(features[\'pitt-train\'].shape[1],5).to(device)\n    optimizer=torch.optim.Adam(head.parameters(),lr=cfg[\'lr\'],weight_decay=cfg[\'weight_decay\'])\n    valx=features[\'pitt-val\']; vyh,vyg=targets(data[\'pitt-val\'])\n    val_targets=[torch.tensor(x,device=device) for x in [vyh,vyg]]\n    step=0; history=[];window=[]\n    total=sum(s for _,s in settings[\'train_plans\'][cfg[\'plan\']])\n    for split,updates in settings[\'train_plans\'][cfg[\'plan\']]:\n        x=features[split]; yh,yg=[torch.tensor(t,device=device) for t in targets(data[split])]\n        for idx in batch_indices(len(x),updates,cfg[\'batch_size\'],generator,device):\n            h,g=task_losses(head(x[idx]),yh[idx],yg[idx]);loss=h+g\n            optimizer.zero_grad(set_to_none=True);loss.backward();optimizer.step()\n            window.append(torch.stack([h.detach(),g.detach()]));step+=1\n            if step%settings[\'eval_every\']==0 or step==total:\n                with torch.inference_mode():\n                    vh,vg=task_losses(head(valx),*val_targets)\n                    probs=head(valx).sigmoid().float().cpu().numpy()\n                train_loss=torch.stack(window).mean(0).cpu().numpy();window=[]\n                hm=hazard_metrics(vyh,probs[:,0],.5,data[\'pitt-val\'].sample_id.to_numpy(),settings[\'k\'])\n                gm,_=group_metrics(vyh,vyg,probs[:,1:],[.5]*4)\n                history.append(dict(step=step,phase=split,hazard_train_loss=float(train_loss[0]),group_train_loss=float(train_loss[1]),hazard_val_loss=float(vh),group_val_loss=float(vg),\n                    hazard_precision_at_k=hm[\'precision_at_k\'],hazard_ba_at_05=hm[\'balanced_accuracy\'],group_macro_ba_at_05=gm[\'balanced_accuracy\']))\n    state={k:v.detach().cpu().clone() for k,v in head.state_dict().items()}\n    return state,predict_head(state,valx),pd.DataFrame(history)\n\n\ndef run_sweep(settings, binding, features, data, run_dir):\n    entries=[];histories=[]; all_cfg=configs(settings)\n    for i,cfg in enumerate(all_cfg):\n        tid=trial_id(cfg);folder=Path(run_dir)/\'trials\'/tid;folder.mkdir(parents=True,exist_ok=True)\n        done=folder/\'complete.json\';start=time.perf_counter()\n        if done.is_file():\n            completion=json.loads(done.read_text()); assert completion[\'binding\']==binding and completion[\'cfg\']==cfg\n            for name,sha in completion[\'file_hashes\'].items():assert file_sha(folder/name)==sha,f\'Damaged trial {tid}: {name}\'\n            payload=torch.load(folder/\'head.pt\',map_location=\'cpu\',weights_only=True)\n            assert payload[\'cfg\']==cfg and payload[\'binding\']==binding\n            state=payload[\'state_dict\'];p=np.load(folder/\'val_probabilities.npz\')[\'probabilities\'];history=pd.read_csv(folder/\'history.csv\');action=\'reused\'\n        else:\n            state,p,history=train_trial(cfg,settings,features,data)\n            save_torch(folder/\'head.pt\',dict(cfg=cfg,binding=binding,state_dict=state,final_step=sum(n for _,n in settings[\'train_plans\'][cfg[\'plan\']]),group_order=GROUPS))\n            save_npz(folder/\'val_probabilities.npz\',probabilities=p,sample_ids=data[\'pitt-val\'].sample_id.to_numpy(dtype=str))\n            save_csv(folder/\'history.csv\',history)\n            save_json(done,dict(cfg=cfg,binding=binding,file_hashes={name:file_sha(folder/name) for name in [\'head.pt\',\'val_probabilities.npz\',\'history.csv\']}));action=\'trained\'\n        cached_ids=np.load(folder/\'val_probabilities.npz\')[\'sample_ids\']\n        assert np.array_equal(cached_ids,data[\'pitt-val\'].sample_id.to_numpy(dtype=str))\n        entries.append(dict(cfg=cfg,trial_id=tid,state_dict=state,val_probabilities=p))\n        histories.append(history.assign(trial_id=tid,**cfg))\n        save_json(Path(run_dir)/\'status.json\',dict(status=\'sweeping\',trials_completed=len(entries),trials_total=len(all_cfg),last_trial=tid))\n        print(f\'[{i+1}/{len(all_cfg)}] {action}: {cfg} ({time.perf_counter()-start:.1f}s)\',flush=True)\n    history=pd.concat(histories,ignore_index=True);save_csv(Path(run_dir)/\'training_history.csv\',history)\n    return entries,history\n\n\ndef cache_features(encoder, df, split, root, run_dir, binding, batch_size, device):\n    cache=Path(run_dir)/\'features\'/(split+\'.pt\')\n    cache_binding={**binding,\'split\':split,\'sample_ids_sha256\':fingerprint(df.sample_id.tolist())}\n    if cache.exists():\n        cached=torch.load(cache,map_location=\'cpu\',weights_only=True)\n        assert cached[\'binding\']==cache_binding,\'Feature cache identity mismatch; start a new run.\'\n        x=cached[\'features\']; assert x.shape==(len(df),encoder.config.hidden_size) and torch.isfinite(x).all()\n        print(\'Reused cached features:\',split)\n        return x.to(device).clone()\n    root=Path(root);mean=torch.tensor(MEAN,device=device).view(1,3,1,1);std=torch.tensor(STD,device=device).view(1,3,1,1)\n    def read(row):\n        payload=(root/row.image_path).read_bytes()\n        assert hashlib.sha256(payload).hexdigest()==row.output_file_sha256,f\'Image checksum mismatch: {row.image_path}\'\n        with Image.open(io.BytesIO(payload)) as im:\n            im.load();assert im.size==(224,224) and im.mode==\'RGB\' and im.format==\'PNG\'\n            return np.array(im,copy=True)\n    pieces=[];start=time.perf_counter()\n    with ThreadPoolExecutor(max_workers=8) as pool,torch.inference_mode():\n        for begin in range(0,len(df),batch_size):\n            pixels=np.stack(list(pool.map(read,df.iloc[begin:begin+batch_size].itertuples(index=False))))\n            x=torch.from_numpy(pixels).permute(0,3,1,2).to(device).float()/255\n            x=(x-mean)/std\n            with torch.autocast(device_type=\'cuda\',dtype=torch.float16,enabled=str(device).startswith(\'cuda\')):\n                feats=encoder(pixel_values=x).last_hidden_state[:,0].float()\n            assert torch.isfinite(feats).all(),\'Nonfinite backbone features; stop rather than saving a bad cache.\'\n            pieces.append(feats.cpu())\n            if begin%(batch_size*20)==0:print(f\'{split}: {min(begin+batch_size,len(df))}/{len(df)} embedded\',flush=True)\n    # clone outside inference_mode so head autograd can save its inputs.\n    output=torch.cat(pieces).clone()\n    save_torch(cache,dict(binding=cache_binding,features=output))\n    print(f\'Cached {split} in {time.perf_counter()-start:.1f}s\')\n    return output.to(device).clone()\n\n\ndef export_selection(selection, entries, schema, binding, run_dir, val_features):\n    root=Path(run_dir)/\'inference\';root.mkdir(parents=True,exist_ok=True)\n    byid={e[\'trial_id\']:e for e in entries}\n    save_json(Path(run_dir)/\'selection.json\',selection)\n    save_json(root/\'selection.json\',selection);save_json(root/\'label_schema.json\',schema)\n    for task,key in [(\'hazard\',\'hazard_deployment_trial\'),(\'tag_groups\',\'group_deployment_trial\')]:\n        chosen=selection[task]\n        for tid in chosen[\'trial_ids\']:\n            e=byid[tid];state=e[\'state_dict\'];indices=slice(0,1) if task==\'hazard\' else slice(1,5)\n            sliced={name:tensor[indices].clone() for name,tensor in state.items()}\n            threshold=chosen[\'summary\'][\'hazard_threshold\'] if task==\'hazard\' else json.loads(chosen[\'summary\'][\'group_thresholds\'])\n            payload=dict(format_version=1,task=task,trial_id=tid,cfg=e[\'cfg\'],binding=binding,state_dict=sliced,thresholds=threshold,\n                         output_labels=[\'hazard\'] if task==\'hazard\' else GROUPS,feature_dim=state[\'weight\'].shape[1],pooling=\'cls\')\n            file=root/\'replicas\'/task/(f\'seed-{e["cfg"]["seed"]}.pt\');save_torch(file,payload)\n            if tid==chosen[key]:save_torch(root/(\'best_\'+task+\'.pt\'),payload)\n    # Reload the deployed heads and verify against the original validation scores.\n    checks={}\n    for task,key,col in [(\'hazard\',\'hazard_deployment_trial\',slice(0,1)),(\'tag_groups\',\'group_deployment_trial\',slice(1,5))]:\n        stored=torch.load(root/(\'best_\'+task+\'.pt\'),map_location=\'cpu\',weights_only=True)\n        e=byid[selection[task][key]];head=nn.Linear(stored[\'feature_dim\'],len(stored[\'output_labels\'])).to(val_features.device)\n        head.load_state_dict(stored[\'state_dict\'])\n        with torch.inference_mode():p=head(val_features).sigmoid().cpu().numpy()\n        original=e[\'val_probabilities\'][:,col]\n        error=float(np.max(np.abs(p-original)));assert error<2e-6,f\'Checkpoint reload error: {task} {error}\'\n        checks[task]=dict(passed=True,max_probability_difference=error,trial_id=stored[\'trial_id\'])\n    save_json(root/\'checkpoint_reload_verification.json\',checks)\n    return root\n\n\ndef prediction_frame(df, hp, gp, ht, gt, **context):\n    cols=[\'sample_id\',\'image_path\',\'split\',\'class\',\'hazard_label\',\'original_class\',\'source_image_path\',\'city\',\'is_manual_collection\']+GROUP_COLUMNS\n    out=df[cols].copy();out[\'p_hazard\']=hp;out[\'pred_hazard\']=(hp>=ht).astype(int)\n    for j,g in enumerate(GROUPS):\n        out[\'p_\'+g]=gp[:,j];out[\'pred_\'+g]=(gp[:,j]>=gt[j]).astype(int)\n        out[\'pipeline_pred_\'+g]=out[\'pred_\'+g]*out[\'pred_hazard\']\n    out[\'hazard_without_group\']=((out.pred_hazard==1)&~(gp>=np.array(gt)).any(1)).astype(int)\n    for name,value in context.items():out[name]=value\n    return out\n\n\ndef evaluate_selected(selection, entries, features, data, run_dir, split):\n    df=data[split];yh,yg=targets(df);byid={e[\'trial_id\']:e for e in entries}\n    rows=[];per_group=[];predictions=[];scores={}\n    for task in [\'hazard\',\'tag_groups\']:\n        chosen=selection[task];ht=chosen[\'summary\'][\'hazard_threshold\'];gt=json.loads(chosen[\'summary\'][\'group_thresholds\'])\n        deployment=chosen[\'hazard_deployment_trial\' if task==\'hazard\' else \'group_deployment_trial\']\n        for tid in chosen[\'trial_ids\']:\n            e=byid[tid];p=predict_head(e[\'state_dict\'],features[split]);scores[tid]=p\n            context=dict(task=task,trial_id=tid,seed=e[\'cfg\'][\'seed\'],config_id=config_id(e[\'cfg\']),deployment_replica=(tid==deployment),split=split)\n            if task==\'hazard\':m=hazard_metrics(yh,p[:,0],ht,df.sample_id.to_numpy(),selection[\'k\'])\n            else:\n                m,groups=group_metrics(yh,yg,p[:,1:],gt)\n                per_group.extend([{**context,**item} for item in groups])\n            rows.append({**context,**m})\n            predictions.append(prediction_frame(df,p[:,0],p[:,1:],ht,gt,**{k:v for k,v in context.items() if k!=\'split\'}))\n    metrics=pd.DataFrame(rows);save_csv(Path(run_dir)/(split+\'_selected_seed_metrics.csv\'),metrics)\n    save_csv(Path(run_dir)/(split+\'_per_group_metrics.csv\'),pd.DataFrame(per_group))\n    save_csv(Path(run_dir)/(split+\'_selected_predictions.csv\'),pd.concat(predictions,ignore_index=True))\n    columns=[c for c in [\'precision_at_k\',\'recall_at_k\',\'balanced_accuracy\',\'precision\',\'recall\',\'f1\',\'exact_match_accuracy\'] if c in metrics]\n    summary=metrics.groupby(\'task\')[columns].agg([\'mean\',\'std\']);atomic_bytes(Path(run_dir)/(split+\'_selected_seed_summary.csv\'),summary.to_csv().encode())\n    hid=selection[\'hazard\'][\'hazard_deployment_trial\'];gid=selection[\'tag_groups\'][\'group_deployment_trial\']\n    hp=scores[hid][:,0];gp=scores[gid][:,1:];ht=selection[\'hazard\'][\'summary\'][\'hazard_threshold\'];gt=json.loads(selection[\'tag_groups\'][\'summary\'][\'group_thresholds\'])\n    deployed=prediction_frame(df,hp,gp,ht,gt,hazard_trial=hid,group_trial=gid)\n    save_csv(Path(run_dir)/(split+\'_deployment_predictions.csv\'),deployed)\n    gm,groups=group_metrics(yh,yg,gp,gt)\n    gated=(gp>=np.array(gt)) & (hp>=ht)[:,None]\n    pm,p_groups=group_metrics(yh,yg,gated.astype(float),[.5]*4,hazard_only=False)\n    report={\'hazard\':hazard_metrics(yh,hp,ht,df.sample_id.to_numpy(),selection[\'k\']),\n            \'groups_on_all_true_hazards\':gm,\'group_metrics_on_all_true_hazards\':groups,\n            \'gated_pipeline_groups_on_all_images\':pm,\'pipeline_group_metrics\':p_groups,\n            \'hazard_trial\':hid,\'group_trial\':gid,\'hazard_threshold\':ht,\'group_thresholds\':gt,\n            \'hazard_without_group_count\':int(deployed.hazard_without_group.sum()),\'test_used_for_selection\':False}\n    save_json(Path(run_dir)/(split+\'_deployment_metrics.json\'),report)\n    return metrics,report\n\n\ndef plot_curves(history, selection, run_dir):\n    import matplotlib.pyplot as plt\n    fig,axs=plt.subplots(2,2,figsize=(13,8))\n    for col,(task,prefix) in enumerate([(\'hazard\',\'hazard\'),(\'tag_groups\',\'group\')]):\n        ids=selection[task][\'trial_ids\'];subset=history[history.trial_id.isin(ids)]\n        for seed,frame in subset.groupby(\'seed\'):\n            axs[0,col].plot(frame.step,frame[prefix+\'_train_loss\'],alpha=.55,label=f\'Train seed {seed}\')\n            axs[0,col].plot(frame.step,frame[prefix+\'_val_loss\'],ls=\'--\',alpha=.55,label=f\'Val seed {seed}\')\n            name=\'hazard_precision_at_k\' if task==\'hazard\' else \'group_macro_ba_at_05\'\n            axs[1,col].plot(frame.step,frame[name],label=f\'Seed {seed}\')\n        axs[0,col].set_title(task+\' — selected configuration\');axs[0,col].set_ylabel(\'BCE loss\');axs[0,col].legend(fontsize=8)\n        axs[1,col].set_ylabel(\'Validation precision@46\' if task==\'hazard\' else \'Validation macro group BA (thresholds 0.5)\')\n        for row in range(2):axs[row,col].set_xlabel(\'Optimizer updates\');axs[row,col].grid(alpha=.2)\n    fig.tight_layout()\n    for ext in [\'png\',\'pdf\']:fig.savefig(Path(run_dir)/(\'selected_learning_curves.\'+ext),dpi=160)\n    plt.show();plt.close(fig)\n'
INFERENCE_SOURCE = '"""Offline inference for a saved sidewalk-data-3 DINOv3 inference/ folder.\nExample: python sidewalk_inference.py --bundle ./inference --images a.jpg b.png --output predictions.csv\n"""\nfrom pathlib import Path\nimport argparse,csv,json\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom PIL import Image,ImageOps\n\n\ndef load_heads(bundle, device):\n    bundle=Path(bundle); heads={}; metadata={}\n    for task in [\'hazard\',\'tag_groups\']:\n        saved=torch.load(bundle/(\'best_\'+task+\'.pt\'),map_location=\'cpu\',weights_only=True)\n        assert saved[\'task\']==task\n        head=nn.Linear(saved[\'feature_dim\'],len(saved[\'output_labels\'])).to(device)\n        head.load_state_dict(saved[\'state_dict\']);head.eval().requires_grad_(False)\n        heads[task]=head;metadata[task]=saved\n    assert metadata[\'hazard\'][\'binding\']==metadata[\'tag_groups\'][\'binding\']\n    return heads,metadata\n\n\nclass SidewalkPredictor:\n    def __init__(self,bundle,device=None):\n        from transformers import AutoModel\n        self.bundle=Path(bundle);self.device=device or (\'cuda\' if torch.cuda.is_available() else \'cpu\')\n        self.settings=json.loads((self.bundle/\'preprocessing.json\').read_text())\n        self.heads,self.metadata=load_heads(self.bundle,self.device)\n        self.encoder=AutoModel.from_pretrained(self.bundle/\'backbone\',local_files_only=True).to(self.device).eval().requires_grad_(False)\n        self.mean=torch.tensor(self.settings[\'mean\'],device=self.device).view(1,3,1,1)\n        self.std=torch.tensor(self.settings[\'std\'],device=self.device).view(1,3,1,1)\n        self.groups=self.metadata[\'tag_groups\'][\'output_labels\']\n        self.hazard_threshold=float(self.metadata[\'hazard\'][\'thresholds\'])\n        self.group_thresholds=np.asarray(self.metadata[\'tag_groups\'][\'thresholds\'])\n        if str(self.device).startswith(\'cuda\'):\n            torch.backends.cuda.matmul.allow_tf32=False\n            torch.backends.cudnn.allow_tf32=False\n\n    @torch.inference_mode()\n    def predict(self,paths,batch_size=64):\n        paths=[Path(p) for p in paths];rows=[]\n        for begin in range(0,len(paths),batch_size):\n            chosen=paths[begin:begin+batch_size];pixels=[]\n            for path in chosen:\n                with Image.open(path) as image:\n                    image=ImageOps.exif_transpose(image).convert(\'RGB\').resize(tuple(self.settings[\'size\']),Image.Resampling.BICUBIC)\n                    pixels.append(np.array(image,copy=True))\n            x=torch.from_numpy(np.stack(pixels)).permute(0,3,1,2).to(self.device).float()/255\n            x=(x-self.mean)/self.std\n            with torch.autocast(device_type=\'cuda\',dtype=torch.float16,enabled=str(self.device).startswith(\'cuda\')):\n                features=self.encoder(pixel_values=x).last_hidden_state[:,0].float()\n            hp=self.heads[\'hazard\'](features).sigmoid().cpu().numpy()[:,0]\n            gp=self.heads[\'tag_groups\'](features).sigmoid().cpu().numpy()\n            for i,path in enumerate(chosen):\n                hazard=bool(hp[i]>=self.hazard_threshold)\n                ungated=[g for j,g in enumerate(self.groups) if gp[i,j]>=self.group_thresholds[j]]\n                row=dict(image_path=str(path),p_hazard=float(hp[i]),pred_hazard=int(hazard),\n                         predicted_class=\'hazard\' if hazard else \'non_hazard\',\n                         predicted_tag_groups=json.dumps(ungated if hazard else []),\n                         ungated_tag_groups=json.dumps(ungated),hazard_without_group=int(hazard and not ungated))\n                row.update({\'p_\'+g:float(gp[i,j]) for j,g in enumerate(self.groups)})\n                rows.append(row)\n        return rows\n\n\ndef main():\n    p=argparse.ArgumentParser(description=__doc__);p.add_argument(\'--bundle\',type=Path,required=True)\n    p.add_argument(\'--images\',nargs=\'+\',required=True);p.add_argument(\'--output\',type=Path,default=Path(\'predictions.csv\'))\n    p.add_argument(\'--device\',default=None);a=p.parse_args()\n    model=SidewalkPredictor(a.bundle,a.device);rows=model.predict(a.images)\n    a.output.parent.mkdir(parents=True,exist_ok=True)\n    with a.output.open(\'w\',newline=\'\') as f:\n        writer=csv.DictWriter(f,fieldnames=list(rows[0]));writer.writeheader();writer.writerows(rows)\n    print(\'Saved\',len(rows),\'predictions to\',a.output)\nif __name__==\'__main__\':main()\n'
import importlib.util, sys
MODULE_PATH = Path('/content/sidewalk_dino_core.py')
MODULE_PATH.write_text(CORE_SOURCE)
spec = importlib.util.spec_from_file_location('sidewalk_dino_core', MODULE_PATH)
core = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = core
spec.loader.exec_module(core)


## Mount Drive and create a persistent run
This must succeed before feature extraction or training. Configuration identity and saved artifacts
are checked when resuming. Tokens are read lazily and are excluded from saved configuration.


In [ ]:
import json, hashlib, shutil, time, tempfile, tarfile, getpass, importlib.metadata
from datetime import datetime, timezone
from uuid import uuid4
import numpy as np, pandas as pd
from google.colab import drive
drive.mount('/content/drive')
assert Path('/content/drive/MyDrive').is_dir()

HF_TOKEN = None
def get_hf_token():
    global HF_TOKEN
    if HF_TOKEN: return HF_TOKEN
    HF_TOKEN = os.environ.get('HF_TOKEN', '').strip()
    if not HF_TOKEN:
        try:
            from google.colab import userdata
            HF_TOKEN = (userdata.get('HF_TOKEN') or '').strip()
        except Exception:
            HF_TOKEN = ''
    if not HF_TOKEN: HF_TOKEN = getpass.getpass('HF read token with DINOv3 access: ').strip()
    if not HF_TOKEN: raise RuntimeError('A token with access to the gated DINOv3 backbone is required.')
    return HF_TOKEN

run_config = dict(version='dino-sidewalk-data-3-v1', settings=SETTINGS,
    backbone=BACKBONE, backbone_revision=BACKBONE_REVISION, pooling='cls', backbone_frozen=True,
    expected_dataset_hashes=EXPECTED_DATASET_HASHES, group_order=core.GROUPS,
    core_code_sha256=hashlib.sha256(CORE_SOURCE.encode()).hexdigest(),
    inference_code_sha256=hashlib.sha256(INFERENCE_SOURCE.encode()).hexdigest(),
    feature_precision='float16', head_precision='float32',
    preprocessing=dict(size=[224,224],resize='full_image_bicubic',mode='RGB',mean=core.MEAN,std=core.STD),
    checkpoint_selection='final_step_only', optimizer='Adam',
    loss='unweighted hazard BCE + mean four-group BCE masked to true hazards')
RUN_DIR = Path(RESUME_RUN_DIR) if RESUME_RUN_DIR else DRIVE_ROOT/(datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'-'+uuid4().hex[:6])
if RESUME_RUN_DIR:
    assert (RUN_DIR/'config.json').is_file(), 'Resume folder has no config.json.'
    assert core.canonical(json.loads((RUN_DIR/'config.json').read_text())) == core.canonical(run_config), 'Code/settings/data changed; start a new run.'
else:
    RUN_DIR.mkdir(parents=True, exist_ok=False)
    core.save_json(RUN_DIR/'config.json',run_config)
probe=RUN_DIR/'.write_probe';probe.write_text('ok');assert probe.read_text()=='ok';probe.unlink()
(RUN_DIR/'sidewalk_dino_core.py').write_text(CORE_SOURCE)
(RUN_DIR/'sidewalk_inference.py').write_text(INFERENCE_SOURCE)
environment=dict(python=sys.version,gpu=torch.cuda.get_device_name(0),torch=str(torch.__version__),
    transformers=transformers.__version__,cuda=str(torch.version.cuda),
    packages={p:importlib.metadata.version(p) for p in ['numpy','pandas','matplotlib','huggingface-hub','Pillow']})
feature_binding=dict(run_config_sha256=core.fingerprint(run_config),backbone_revision=BACKBONE_REVISION,
    gpu=environment['gpu'],torch=environment['torch'],transformers=environment['transformers'],precision='float16',
    torch_tf32=False,cache_batch_size=CACHE_BATCH_SIZE)
if (RUN_DIR/'feature_binding.json').exists():
    assert json.loads((RUN_DIR/'feature_binding.json').read_text())==feature_binding, 'Feature-extraction environment changed; start a new run.'
else:core.save_json(RUN_DIR/'feature_binding.json',feature_binding)
core.save_json(RUN_DIR/'environment.json',environment)
assert len(core.configs(SETTINGS))==84
print('Persistent run:',RUN_DIR)
print('84 final-checkpoint trials; selection uses validation only.')


## Load and verify dataset 3
Extraction and feature reads use Colab's local disk. The server archive and manifests are pinned by
SHA-256. Each image is checked against its recorded hash when features are first computed. All
277 additional Pittsburgh training screenshots are included as non-hazards. Existing annotation
conflicts are retained under dataset 3's documented policy; no silent filtering or deduplication occurs.


In [ ]:
from huggingface_hub import HfApi, hf_hub_download
already_extracted = all((DATA_ROOT/name).is_file() for name in EXPECTED_DATASET_HASHES)
if not already_extracted:
    if DATA_ROOT.exists(): raise RuntimeError(f'{DATA_ROOT} is incomplete; choose a fresh DATA_ROOT or remove your incomplete extraction.')
    if DATA_SOURCE == 'drive':
        if not DRIVE_ARCHIVE.is_file():
            raise FileNotFoundError(f'Upload /data/datasets/sidewalk-data-3.tar from the server to {DRIVE_ARCHIVE}, then rerun this cell.')
        local_tar=Path('/content/sidewalk-data-3.tar')
        if not local_tar.exists() or core.file_sha(local_tar)!=EXPECTED_ARCHIVE_SHA256:
            shutil.copyfile(DRIVE_ARCHIVE, local_tar)
        download_info=dict(source='google_drive',archive=str(DRIVE_ARCHIVE))
    elif DATA_SOURCE == 'huggingface':
        assert HF_DATA_REPO, 'Set HF_DATA_REPO to the repository containing sidewalk-data-3.tar.'
        revision=HF_DATA_REVISION or HfApi(token=get_hf_token()).dataset_info(HF_DATA_REPO).sha
        local_tar=Path(hf_hub_download(HF_DATA_REPO,HF_DATA_ARCHIVE,repo_type='dataset',revision=revision,token=get_hf_token()))
        download_info=dict(source='huggingface',repo=HF_DATA_REPO,revision=revision,archive=HF_DATA_ARCHIVE)
    else:
        raise FileNotFoundError('DATA_SOURCE=existing requires the fully extracted dataset at DATA_ROOT.')
    assert core.file_sha(local_tar)==EXPECTED_ARCHIVE_SHA256, 'Archive is not the prepared dataset-3 snapshot.'
    DATA_ROOT.parent.mkdir(parents=True,exist_ok=True)
    with tempfile.TemporaryDirectory(dir=DATA_ROOT.parent,prefix='sidewalk3-extract-') as temp:
        with tarfile.open(local_tar) as archive:
            for member in archive.getmembers():
                path=Path(member.name)
                assert not path.is_absolute() and '..' not in path.parts and path.parts[0]=='sidewalk-data-3'
                assert member.isfile() or member.isdir(), 'Archive contains a non-regular entry.'
            archive.extractall(temp,filter='data')
        Path(temp,'sidewalk-data-3').rename(DATA_ROOT)
    core.save_json(RUN_DIR/'download.json',{**download_info,'sha256':EXPECTED_ARCHIVE_SHA256})

data,schema=core.load_data(DATA_ROOT,EXPECTED_DATASET_HASHES)
assert {s:len(df) for s,df in data.items()}=={'other-cities-train':22912,'pitt-train':1276,'pitt-val':382,'pitt-test':380}
snapshot=RUN_DIR/'dataset_metadata';snapshot.mkdir(exist_ok=True)
for name in list(EXPECTED_DATASET_HASHES)+['README.md','LABELING_RULES.md','PREPROCESSING.md','build_config.json','tag_mapping.csv']:
    shutil.copy2(DATA_ROOT/name,snapshot/name)
for folder in ['statistics','audit']:
    for name in (['binary_class_counts.csv','tag_group_counts.csv','tag_counts_by_split.csv'] if folder=='statistics' else ['duplicate_label_summary.json','release_validation.json','label_review_candidates.csv']):
        shutil.copy2(DATA_ROOT/folder/name,snapshot/name)
summary=pd.DataFrame([dict(split=s,images=len(df),hazard=int(df.hazard_label.sum()),non_hazard=int((df.hazard_label==0).sum()),manual_nulls=int(df.is_manual_collection.sum())) for s,df in data.items()])
core.save_csv(RUN_DIR/'dataset_summary.csv',summary)
display(summary)
print('Group output order:',core.GROUPS)


## Frozen backbone and cached training/validation features
The encoder uses the exact pinned DINOv3 revision. Features use FP16 on GPU; linear heads train
in FP32. Images are already RGB 224×224, so only ImageNet normalization is applied here—no extra
crop or resize. The frozen backbone is saved once to Drive for later offline inference.
Test features are deliberately deferred until model selection has finished.


In [ ]:
from transformers import AutoModel
encoder=AutoModel.from_pretrained(BACKBONE,revision=BACKBONE_REVISION,token=get_hf_token()).to('cuda').eval().requires_grad_(False)
assert not any(p.requires_grad for p in encoder.parameters())
features={}
for split in ['other-cities-train','pitt-train','pitt-val']:
    features[split]=core.cache_features(encoder,data[split],split,DATA_ROOT,RUN_DIR,feature_binding,CACHE_BATCH_SIZE,'cuda')

inference_dir=RUN_DIR/'inference';inference_dir.mkdir(exist_ok=True)
backbone_dir=inference_dir/'backbone'
if backbone_dir.exists():
    assert (inference_dir/'backbone_files.json').exists(), 'Incomplete backbone export.'
    for relative,expected in json.loads((inference_dir/'backbone_files.json').read_text()).items():
        assert core.file_sha(backbone_dir/relative)==expected
else:
    pending=inference_dir/'.backbone_pending'
    if pending.exists(): shutil.rmtree(pending)  # only this run's interrupted export
    with tempfile.TemporaryDirectory() as temp:
        encoder.save_pretrained(temp,safe_serialization=True)
        shutil.copytree(temp,pending)
    hashes={str(p.relative_to(pending)):core.file_sha(p) for p in pending.rglob('*') if p.is_file()}
    core.save_json(inference_dir/'backbone_files.json',hashes)
    pending.rename(backbone_dir)
print('Backbone and feature caches saved on Drive.')


## Run the 84 trials
The two training orders share the same settings. Other→Pitt continues the same head and Adam
optimizer; it does not reset either at the phase boundary. Loss curves are recorded every 250
updates. These intermediate metrics use 0.5 for display only; selection searches the configured
threshold grid after training. Only final-update checkpoints compete for selection.


In [ ]:
try:
    entries,history=core.run_sweep(SETTINGS,feature_binding,features,data,RUN_DIR)
except BaseException as error:
    core.save_json(RUN_DIR/'status.json',dict(status='interrupted',error_type=type(error).__name__,message=str(error)))
    raise
print('All trials are saved:',RUN_DIR/'trials')


## Select the two configurations on validation
Precision ties are determined using integer summed top-46 hits across seeds, avoiding accidental
floating-point tie-breaking. Balanced accuracy breaks these ties after threshold selection.
Remaining exact ties prefer Pitt-only, then lower learning rate, then lower weight decay.
Exact threshold ties choose the lower threshold. Group selection is independent of hazard selection.

Four threshold searches are sufficient for the group task: macro balanced accuracy is the sum of
four independent per-group scores. This is equivalent to checking all 3⁴ threshold combinations
for the selected objective, without changing training or using test data.


In [ ]:
selection,validation_summary,hazard_thresholds,group_thresholds=core.select_models(
    entries,data['pitt-val'],SETTINGS['thresholds'],SETTINGS['seeds'],SETTINGS['k'])
core.save_json(RUN_DIR/'selection.json',selection)
core.save_csv(RUN_DIR/'validation_configurations.csv',validation_summary)
core.save_csv(RUN_DIR/'validation_hazard_thresholds.csv',hazard_thresholds)
core.save_csv(RUN_DIR/'validation_group_thresholds.csv',group_thresholds)
frames=[]
for entry in entries:
    frame=data['pitt-val'][['sample_id','image_path','hazard_label','original_class']+core.GROUP_COLUMNS].copy()
    frame['trial_id']=entry['trial_id'];frame['config_id']=core.config_id(entry['cfg']);frame['seed']=entry['cfg']['seed']
    frame['p_hazard']=entry['val_probabilities'][:,0]
    for j,g in enumerate(core.GROUPS):frame['p_'+g]=entry['val_probabilities'][:,j+1]
    frames.append(frame)
core.save_csv(RUN_DIR/'validation_all_predictions.csv',pd.concat(frames,ignore_index=True))
print('All 28 configurations at their selected validation thresholds:')
display(validation_summary.sort_values(['precision_at_k_mean','hazard_ba_mean'],ascending=False))
print('Chosen hazard configuration:',selection['hazard'])
print('Chosen tag-group configuration:',selection['tag_groups'])


## Save deployable weights and verify loading
`inference/best_hazard.pt` and `inference/best_tag_groups.pt` contain the independently selected
heads, thresholds and label order. `inference/backbone/` contains the complete frozen encoder.
All three winning-configuration replicas are retained under `inference/replicas/`.
The exported heads are reloaded and checked against the original validation probabilities before
test evaluation. The chosen single replicas and the three-seed configuration means are different
quantities and are reported separately.


In [ ]:
inference_dir=core.export_selection(selection,entries,schema,feature_binding,RUN_DIR,features['pitt-val'])
(inference_dir/'sidewalk_inference.py').write_text(INFERENCE_SOURCE)
core.save_json(inference_dir/'preprocessing.json',dict(size=[224,224],resize='full_image_bicubic',exif_transpose=True,
    mode='RGB',scale=255,mean=core.MEAN,std=core.STD,pooling='cls',gpu_precision='float16',head_precision='float32',
    backbone=BACKBONE,backbone_revision=BACKBONE_REVISION))
requirements='\n'.join(['torch=='+str(torch.__version__).split('+')[0], 'transformers=='+transformers.__version__,
    'numpy=='+importlib.metadata.version('numpy'),'Pillow=='+importlib.metadata.version('Pillow')])+'\n'
(inference_dir/'requirements.txt').write_text(requirements)
(inference_dir/'README.md').write_text('# Saved sidewalk-data-3 models\n\nKeep this entire inference/ directory. No Hugging Face token or network download is needed\nwhen loading the complete bundle: backbone/ contains the frozen encoder weights.\n\nRun `python sidewalk_inference.py --bundle . --images image1.png image2.jpg --output predictions.csv`.\nInstall requirements.txt using a PyTorch build appropriate for the inference device.\n\nbest_hazard.pt and best_tag_groups.pt are the independently selected single deployment\nreplicas from the validation-winning configurations. selection.json records all hyperparameters,\nseeds and thresholds. replicas/ retains all three seeds of both selected configurations.\nReported three-seed means are not ensemble metrics or the single-model deployment scores.\nNo test score was used to choose these files. checkpoint_reload_verification.json verifies the\nexported head probabilities on the original validation features.\n\nInput: any ordinary image; EXIF orientation, RGB conversion, full-frame bicubic resize to\n224x224, and saved ImageNet normalization are applied identically to the dataset pipeline.\nFor already processed dataset images the resize is an identity operation.\n\nOutputs: p_hazard, binary decision, four conditional tag-group probabilities and multiple\ngroup decisions. Group probabilities need not sum to one. Pipeline group decisions are\nsuppressed for a predicted non-hazard. If a hazard has no group above its threshold,\nhazard_without_group=1; no arbitrary argmax or forced group is introduced.\nGroup evaluation in the experiment uses ALL true hazards, regardless of stage 1 predictions.\nThe separate gated pipeline metrics use all images and include stage 1 errors.\n\nCPU inference uses float32 backbone operations, while the T4 experiment uses float16 features;\nsmall numerical differences across hardware/precision are possible. Model weights and the\ncomplete preprocessing specification are saved for reproducibility.\n')
core.save_json(RUN_DIR/'status.json',dict(status='selected_models_exported',trials_completed=len(entries)))
print('Inference weights:',inference_dir)
print((inference_dir/'checkpoint_reload_verification.json').read_text())


## Validation summaries and learning curves
The configuration table above uses optimized validation thresholds. The subtype learning curve
below uses 0.5 only as a consistent training-time diagnostic; it is not the selection score.


In [ ]:
validation_metrics,validation_deployment=core.evaluate_selected(selection,entries,features,data,RUN_DIR,'pitt-val')
display(validation_metrics)
print('Single deployment-model validation scores:')
print('Hazard:',validation_deployment['hazard'])
print('Groups:',validation_deployment['groups_on_all_true_hazards'])
core.plot_curves(history,selection,RUN_DIR)


## Test the frozen selections
No threshold, configuration, seed, or checkpoint is selected here. Save per-image probabilities,
per-group precision/recall/F1/balanced accuracy, confusion counts, the three-seed summaries, and
separate single-deployment-model and gated-pipeline metrics. The test labels are the new dataset-3
labels, so the old report's test scores and old two-subtype definition do not apply.


In [ ]:
selection_hash=core.file_sha(RUN_DIR/'selection.json')
if RUN_TEST:
    selection=json.loads((RUN_DIR/'selection.json').read_text())
    features['pitt-test']=core.cache_features(encoder,data['pitt-test'],'pitt-test',DATA_ROOT,RUN_DIR,feature_binding,CACHE_BATCH_SIZE,'cuda')
    test_metrics,test_deployment=core.evaluate_selected(selection,entries,features,data,RUN_DIR,'pitt-test')
    assert core.file_sha(RUN_DIR/'selection.json')==selection_hash
    display(test_metrics)
    print('Single deployment-model test hazard scores:',test_deployment['hazard'])
    print('Single deployment-model test tag-group scores:',test_deployment['groups_on_all_true_hazards'])
else:
    print('Test disabled. The selected inference weights are already saved.')


## Check offline image inference and finish
Reload the exported bundle and predict a few images without contacting Hugging Face. This cell
checks the inference entry point; it does not change selection. Keep the entire Drive run folder,
especially `inference/`, for later inference and reporting.


In [ ]:
spec=importlib.util.spec_from_file_location('sidewalk_inference',inference_dir/'sidewalk_inference.py')
infer_module=importlib.util.module_from_spec(spec);spec.loader.exec_module(infer_module)
# Free the training encoder before loading its saved offline copy.
del encoder
torch.cuda.empty_cache()
predictor=infer_module.SidewalkPredictor(inference_dir,device='cuda')
example_split='pitt-test' if RUN_TEST else 'pitt-val'
examples=[DATA_ROOT/p for p in data[example_split].image_path.iloc[:3]]
preview=pd.DataFrame(predictor.predict(examples))
core.save_csv(RUN_DIR/'offline_inference_example.csv',preview)
display(preview)
core.save_json(RUN_DIR/'status.json',dict(status='complete' if RUN_TEST else 'complete_without_test',
    trials_completed=len(entries),inference_bundle=str(inference_dir),offline_inference_checked=True,test_used_for_selection=False))
# A compact checksum manifest covers all saved inference files, including encoder weights.
core.save_json(inference_dir/'files_sha256.json',{str(p.relative_to(inference_dir)):core.file_sha(p)
    for p in inference_dir.rglob('*') if p.is_file() and p.name!='files_sha256.json'})
print('Finished. Complete run:',RUN_DIR)
print('Deployable models:',inference_dir)


## Saved outputs

- `config.json`, `environment.json`, `feature_binding.json`, `dataset_metadata/`: settings and data identity.
- `trials/`: all 84 final weights, validation probabilities, and learning histories; resumable.
- `validation_configurations.csv`: all 28 configurations with their selected thresholds and seed means/std.
- `validation_hazard_thresholds.csv`, `validation_group_thresholds.csv`: every threshold and seed score.
- `selection.json`: exact selected configurations, thresholds, replicas, and selection rules.
- `pitt-val_*`, `pitt-test_*`: per-image predictions, per-group metrics, seed summaries and deployment metrics.
- `selected_learning_curves.png/.pdf`: loss and validation metric curves for the winning configurations.
- **`inference/`**: complete backbone, **`best_hazard.pt`**, **`best_tag_groups.pt`**, all selected seed weights,
  thresholds, label schema, preprocessing, checksums, and a standalone offline inference script.

For the group task, “macro balanced accuracy” averages sensitivity and specificity within each
of the four binary group decisions, then averages equally across groups. It is not four-way
multiclass accuracy or macro positive recall. The evaluated population is all true hazards.
[DINOv3 API documentation](https://huggingface.co/docs/transformers/model_doc/dinov3) ·
[Balanced accuracy definition](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.balanced_accuracy_score.html)
